<a href="https://colab.research.google.com/github/omark243/Big-Data-/blob/main/Assignment%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pyspark gdown

In [ ]:
import os, glob

folder_url = "https://drive.google.com/drive/folders/1_qlhp6XK8xP4fenJZTU52Ex_cc8JF8RW?usp=sharing"

!mkdir -p /content/assignment2_data
!gdown --folder "$folder_url" -O /content/assignment2_data --remaining-ok

csv_files = glob.glob("/content/assignment2_data/**/*.csv", recursive=True)

print("CSV files found:")
for f in csv_files:
    print(f)

csv_path = csv_files[0]
print("Using:", csv_path)

Retrieving folder contents
Processing file 12HUsOFZtowRrG5aqOQ-oDr0oihSHVeBf student_performance_large.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=12HUsOFZtowRrG5aqOQ-oDr0oihSHVeBf
To: /content/assignment2_data/student_performance_large.csv
100% 1.89k/1.89k [00:00<00:00, 5.79MB/s]
Download completed
CSV files found:
/content/assignment2_data/student_performance_large.csv
Using: /content/assignment2_data/student_performance_large.csv


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Assignment2_Students") \
    .getOrCreate()

df = spark.read.csv(csv_path, header=True, inferSchema=True)

df.show(5)
df.printSchema()
print(df.columns)

+-----------+----------+-----------+-----------+------+
|study_hours|attendance|assignments|sleep_hours|result|
+-----------+----------+-----------+-----------+------+
|          2|        41|          5|          5|     0|
|          4|        48|          2|          8|     0|
|          2|        77|          7|          4|     1|
|          1|        45|          4|          5|     0|
|          9|        78|          1|          8|     1|
+-----------+----------+-----------+-----------+------+
only showing top 5 rows
root
 |-- study_hours: integer (nullable = true)
 |-- attendance: integer (nullable = true)
 |-- assignments: integer (nullable = true)
 |-- sleep_hours: integer (nullable = true)
 |-- result: integer (nullable = true)

['study_hours', 'attendance', 'assignments', 'sleep_hours', 'result']


In [ ]:
for old_col in df.columns:
    new_col = old_col.strip().lower().replace(" ", "_")
    df = df.withColumnRenamed(old_col, new_col)

df.show(5)
print(df.columns)

+-----------+----------+-----------+-----------+------+
|study_hours|attendance|assignments|sleep_hours|result|
+-----------+----------+-----------+-----------+------+
|          2|        41|          5|          5|     0|
|          4|        48|          2|          8|     0|
|          2|        77|          7|          4|     1|
|          1|        45|          4|          5|     0|
|          9|        78|          1|          8|     1|
+-----------+----------+-----------+-----------+------+
only showing top 5 rows
['study_hours', 'attendance', 'assignments', 'sleep_hours', 'result']


In [ ]:
total_students = df.count()

print("1) Total number of students =", total_students)

1) Total number of students = 150


In [ ]:
from pyspark.sql.functions import avg

avg_study_hours = df.select(avg("study_hours")).collect()[0][0]

print("2) Average study_hours =", avg_study_hours)

2) Average study_hours = 5.18


In [ ]:
students_more_than_5 = df.filter(df.study_hours > 5).count()

print("3) Students studied more than 5 hours =", students_more_than_5)

3) Students studied more than 5 hours = 67


In [ ]:
attendance_less_than_60 = df.filter(df.attendance < 60).count()

print("4) Students with attendance less than 60 =", attendance_less_than_60)

4) Students with attendance less than 60 = 51


In [ ]:
print("5) Passed and failed students:")

df.groupBy("result").count().show()

5) Passed and failed students:
+------+-----+
|result|count|
+------+-----+
|     1|  126|
|     0|   24|
+------+-----+



In [ ]:
from pyspark.sql.functions import col, lower, trim, when

df_model = df.withColumn(
    "label",
    when(lower(trim(col("result").cast("string"))).isin("pass", "passed", "1", "1.0"), 1.0)
    .when(lower(trim(col("result").cast("string"))).isin("fail", "failed", "0", "0.0"), 0.0)
)

df_model = df_model.dropna(subset=["study_hours", "attendance", "assignments", "sleep_hours", "label"])

df_model.select("study_hours", "attendance", "assignments", "sleep_hours", "result", "label").show()

+-----------+----------+-----------+-----------+------+-----+
|study_hours|attendance|assignments|sleep_hours|result|label|
+-----------+----------+-----------+-----------+------+-----+
|          2|        41|          5|          5|     0|  0.0|
|          4|        48|          2|          8|     0|  0.0|
|          2|        77|          7|          4|     1|  1.0|
|          1|        45|          4|          5|     0|  0.0|
|          9|        78|          1|          8|     1|  1.0|
|          4|        85|          9|          7|     1|  1.0|
|          4|        68|         10|          6|     1|  1.0|
|          1|        88|          3|          7|     0|  0.0|
|          6|        57|          3|          5|     1|  1.0|
|          6|        46|          2|          7|     1|  1.0|
|          2|        62|          6|          8|     1|  1.0|
|          5|        91|          1|          7|     1|  1.0|
|          9|        47|          7|          4|     1|  1.0|
|       

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

feature_cols = ["study_hours", "attendance", "assignments", "sleep_hours"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

data = assembler.transform(df_model)

model_data = data.select("features", "label", "result")

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction"
)

model = lr.fit(model_data)

print("6) Model trained successfully.")

6) Model trained successfully.


In [ ]:
predictions = model.transform(model_data)

print("7) Predictions:")

predictions.select("result", "prediction").show()

7) Predictions:
+------+----------+
|result|prediction|
+------+----------+
|     0|       0.0|
|     0|       0.0|
|     1|       1.0|
|     0|       0.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     0|       0.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     0|       0.0|
+------+----------+
only showing top 20 rows


In [ ]:
correct_predictions = predictions.filter(col("label") == col("prediction")).count()
total_predictions = predictions.count()

accuracy = correct_predictions / total_predictions

print("8) Number of correct predictions =", correct_predictions)
print("Total predictions =", total_predictions)
print("Accuracy =", accuracy)

8) Number of correct predictions = 150
Total predictions = 150
Accuracy = 1.0
